In [28]:
import os
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

In [ ]:
data_dir = '/group/glastonbury/GWAS/F20208v3_DiffAE/select_latents_r80/nNs_Qntl_INF30_DiffAE128_5Sd_r80_discov_fullDSV3/latents_collect/'
all_files = [f for f in os.listdir(data_dir) if f.endswith('_processed.tsv')]
data_list = []
for file in all_files:
    file_path = os.path.join(data_dir, file)
    data = pd.read_csv(file_path, sep='\t')
    data.set_index(['IID','FID'], inplace=True)
    data_list.append(data)
all_latents = pd.concat(data_list, axis=1)
X = all_latents.values

In [19]:
selected_latents = pd.read_table("/group/glastonbury/GWAS/F20208v3_DiffAE/select_latents_r80/nNs_Qntl_INF30_DiffAE128_5Sd_r80_discov_fullDSV3/processed.tsv")
selected_latents.set_index(['IID','FID'], inplace=True)
Z = selected_latents.values

In [ ]:
assert all_latents.index.equals(selected_latents.index)

In [24]:
# X: N x 640 (original 5-run latents)
# Z: N x 182 (consensus latents)


model = LinearRegression()
model.fit(Z, X) # Multivariate regression, fits all 640 outputs at once

X_hat = model.predict(Z)

# Variance explained per latent
R2_per_latent = 1 - np.var(X - X_hat, axis=0) / np.var(X, axis=0)

# Total variance explained
total_R2 = 1 - np.var(X - X_hat) / np.var(X)

total_R2

0.9801095524486192

In [25]:
model = LinearRegression()
model.fit(Z, X)
X_hat = model.predict(Z)

# Residual Sum of Squares (sum over all elements)
ss_res = np.sum((X - X_hat) ** 2)

# Total Sum of Squares (sum of squared deviations from COLUMN means). We subtract the mean of each column, not the global mean
ss_tot = np.sum((X - np.mean(X, axis=0)) ** 2)

total_R2 = 1 - (ss_res / ss_tot)

total_R2

0.9801095523625765

In [27]:
model = LinearRegression()
model.fit(Z, X) # Multivariate regression, fits all 640 outputs at once
X_hat = model.predict(Z)
r2_score(X, X_hat, multioutput='variance_weighted')


0.9801095523625765